# 04 Final Core Hybrid Ranking - Two Models

Run all cells once.


In [ ]:
# 04_final_core_hybrid_ranking_TWO_MODELS.py
# Offline ranking experiment aligned with backend core recommendation logic.
#
# This script builds TWO scores:
#   Model 1: Skill-only Baseline
#       skill_only_score = skill_overlap_score
#   Model 2: Core Hybrid Recommendation Model
#       taxonomy_score = 0.65 * skill_overlap_score + 0.25 * group_similarity_score + 0.10 * dominant_group_score
#       core_recommendation_score = 0.70 * taxonomy_score + 0.30 * semantic_score_norm
#       offline_final_score = core_recommendation_score
#
# Why offline_final_score does not include context scores:
#   The recommendation offline dataset does not contain backend context fields such as
#   location, job type, position level, or experience level. Therefore offline experiment
#   evaluates the core AI recommendation score: Skill + Taxonomy + Semantic.
#   Backend can extend this score with context fields when they are available.

from __future__ import annotations

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# =========================
# Config
# =========================
TAXONOMY_WEIGHTS = { # tính taxonomy_score từ skill/group signals. Tổng bằng 1.
    'skill_overlap_score': 0.65, # Vì tuyển dụng IT vẫn phụ thuộc nhiều vào skill cụ thể.
    'group_similarity_score': 0.25, # Group similarity cũng quan trọng nhưng có thể kém chính xác hơn skill overlap; Vì taxonomy giúp nhận ra sự liên quan theo nhóm.
    'dominant_group_score': 0.10, # Vì nhóm chính giống nhau là tín hiệu bổ sung tổng quát hơn, không chi tiết bằng skill cụ thể.
}

CORE_WEIGHTS = { # Sau khi đã có taxonomy_score, mình kết hợp thêm embedding semantic:
    'taxonomy_score': 0.70, # 0.70 * taxonomy_score
    'semantic_score_norm': 0.30, # + 0.30 * semantic_score_norm
}

OUTPUT_BASENAME = '13_candidate_job_hybrid_ranking'

# =========================
# Path helpers
# =========================
def here() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd().resolve()


def candidate_roots() -> list[Path]:
    cwd = here()
    roots = [cwd, cwd.parent, cwd.parent.parent, Path.cwd().resolve()]
    result: list[Path] = []
    for p in roots:
        if p.exists() and p not in result:
            result.append(p)
    return result


def find_first(patterns: list[str], roots: list[Path]) -> Path:
    hits: list[Path] = []
    for root in roots:
        for pattern in patterns:
            hits.extend(root.rglob(pattern))

    clean_hits: list[Path] = []
    for p in hits:
        parts = set(p.parts)
        if any(part.startswith('.') for part in p.parts):
            continue
        if '.venv' in parts or 'node_modules' in parts:
            continue
        clean_hits.append(p)

    if not clean_hits:
        raise FileNotFoundError('Cannot find any file matching: ' + ', '.join(patterns))

    def score_path(p: Path) -> tuple[int, int, str]:
        s = str(p).lower()
        priority = 0
        if 'data_outputs' in s or 'outputs' in s:
            priority -= 10
        if p.suffix.lower() == '.parquet':
            priority -= 2
        if p.suffix.lower() == '.xlsx':
            priority -= 1
        return (priority, len(p.parts), str(p))

    return sorted(clean_hits, key=score_path)[0]


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Unsupported input file type: {path}')


def ensure_output_dir() -> Path:
    cwd = here()
    preferred = cwd.parent / 'data_outputs'
    if preferred.exists() or cwd.name.lower() in {'notebooks', 'embedding'}:
        preferred.mkdir(parents=True, exist_ok=True)
        return preferred
    fallback = cwd / 'data_outputs'
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback

# =========================
# Load data
# =========================
roots = candidate_roots()
print('Searching input files from roots:')
for r in roots:
    print(' -', r)

matching_path = find_first([ # output từ ProcessPipeline
    '*candidate_job_matching*.parquet',
    '*candidate_job_matching*.xlsx',
    '*candidate_job_match*.parquet',
    '*candidate_job_match*.xlsx',
], roots)

semantic_path = find_first([ # output từ Embedding file 03
    '*candidate_job_semantic_similarity*.parquet',
    '*candidate_job_semantic_similarity*.xlsx',
    '*semantic_similarity*.parquet',
    '*semantic_similarity*.xlsx',
], roots)

print('\nInput matching file:', matching_path)
print('Input semantic file:', semantic_path)

matching_df = read_table(matching_path)
semantic_df = read_table(semantic_path)

matching_df.columns = [str(c).strip() for c in matching_df.columns]
semantic_df.columns = [str(c).strip() for c in semantic_df.columns]

required_matching = [
    'candidate_id', 'job_id',
    'skill_overlap_score', 'group_similarity_score', 'dominant_group_score'
]
missing = [c for c in required_matching if c not in matching_df.columns]
if missing:
    raise KeyError(f'Matching file is missing required columns: {missing}. Available: {list(matching_df.columns)}')

required_semantic = ['candidate_id', 'job_id', 'semantic_similarity']
missing = [c for c in required_semantic if c not in semantic_df.columns]
if missing:
    raise KeyError(f'Semantic file is missing required columns: {missing}. Available: {list(semantic_df.columns)}')

# Keep old ProcessPipeline final_score for reference only.
if 'final_score' in matching_df.columns and 'pipeline_final_score' not in matching_df.columns:
    matching_df = matching_df.rename(columns={'final_score': 'pipeline_final_score'})
if 'baseline_score' in matching_df.columns and 'pipeline_final_score' not in matching_df.columns:
    matching_df = matching_df.rename(columns={'baseline_score': 'pipeline_final_score'})

semantic_keep = ['candidate_id', 'job_id', 'semantic_similarity']
for optional in ['semantic_rank', 'candidate_text', 'job_text']:
    if optional in semantic_df.columns:
        semantic_keep.append(optional)
semantic_df = semantic_df[semantic_keep].drop_duplicates(['candidate_id', 'job_id'])

# =========================
# Merge matching + semantic
# =========================
df = matching_df.merge(semantic_df, on=['candidate_id', 'job_id'], how='left')
if df['semantic_similarity'].isna().any():
    n_missing = int(df['semantic_similarity'].isna().sum())
    print(f'Warning: {n_missing} rows have missing semantic_similarity after merge. They will be filled with 0 after normalization.')

# =========================
# Score computation
# =========================
for col in ['skill_overlap_score', 'group_similarity_score', 'dominant_group_score', 'semantic_similarity']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# Model 1: Skill-only baseline.
# This is the simplest baseline: it does not use taxonomy and does not use embedding.
df['skill_only_score'] = df['skill_overlap_score']

# CHUẨN BỊ NGUYÊN LIỆU CHO MODEL 2
# Taxonomy score: same core baseline as ProcessPipeline / backend.
# Skill overlap is the direct skill signal.
# Group similarity + dominant group are taxonomy signals.
df['taxonomy_score'] = (
    TAXONOMY_WEIGHTS['skill_overlap_score'] * df['skill_overlap_score']
    + TAXONOMY_WEIGHTS['group_similarity_score'] * df['group_similarity_score']
    + TAXONOMY_WEIGHTS['dominant_group_score'] * df['dominant_group_score']
)

# Normalize semantic similarity to 0-1. chuẩn hoá về thang 0 - 1 
# Prefer min-max normalization on the evaluated table because cosine/e5 scores may not be naturally 0-1.
sem_min = df['semantic_similarity'].min()
sem_max = df['semantic_similarity'].max()
if sem_max > sem_min:
    df['semantic_score_norm'] = (df['semantic_similarity'] - sem_min) / (sem_max - sem_min)
else:
    df['semantic_score_norm'] = 0.0

df['semantic_score_norm'] = df['semantic_score_norm'].fillna(0.0)

# Model 2: Core Hybrid Recommendation Model.
# Offline recommendation does not have context fields yet, so this is the final score for offline metrics.
df['core_recommendation_score'] = (
    CORE_WEIGHTS['taxonomy_score'] * df['taxonomy_score']
    + CORE_WEIGHTS['semantic_score_norm'] * df['semantic_score_norm']
)

df['offline_final_score'] = df['core_recommendation_score']

# Compatibility aliases for old notebooks / UI exports.
df['hybrid_score'] = df['core_recommendation_score']
df['final_score'] = df['offline_final_score']

# Recommendation ranks for both models.
# These are Top-K recommendation ranks, not job seniority/position level.
df = df.sort_values(['candidate_id', 'offline_final_score'], ascending=[True, False]).reset_index(drop=True)

# model 1
df['rank_skill_only'] = (
    df.groupby('candidate_id')['skill_only_score']
    .rank(method='first', ascending=False)
    .astype(int)
)

# model 2
df['rank_core_hybrid'] = (
    df.groupby('candidate_id')['offline_final_score']
    .rank(method='first', ascending=False)
    .astype(int)
)

# Main model rank used by UI/backward-compatible notebooks.
df['final_rank'] = df['rank_core_hybrid']
df['hybrid_rank'] = df['rank_core_hybrid']
df['recommendation_rank'] = df['rank_core_hybrid']

# Sort final output by the deployed/offline main model.
df = df.sort_values(['candidate_id', 'final_rank'], ascending=[True, True]).reset_index(drop=True)

# =========================
# Output
# =========================
output_dir = ensure_output_dir()
parquet_path = output_dir / f'{OUTPUT_BASENAME}.parquet'
xlsx_path = output_dir / f'{OUTPUT_BASENAME}.xlsx'

front_cols = [
    'candidate_id', 'job_id',
    'skill_overlap_score', 'group_similarity_score', 'dominant_group_score',
    'skill_only_score', 'taxonomy_score',
    'semantic_similarity', 'semantic_score_norm',
    'core_recommendation_score', 'offline_final_score', 'final_score',
    'rank_skill_only', 'rank_core_hybrid', 'final_rank', 'recommendation_rank',
]
if 'pipeline_final_score' in df.columns:
    front_cols.insert(8, 'pipeline_final_score')
remaining_cols = [c for c in df.columns if c not in front_cols]
df = df[[c for c in front_cols if c in df.columns] + remaining_cols]

df.to_parquet(parquet_path, index=False)
df.to_excel(xlsx_path, index=False)

print('\nSaved:')
print(' -', parquet_path)
print(' -', xlsx_path)
print('\nRows:', len(df))
print('Candidates:', df['candidate_id'].nunique())
print('Jobs:', df['job_id'].nunique())

summary_cols = [
    'skill_only_score', 'taxonomy_score', 'semantic_score_norm',
    'core_recommendation_score', 'offline_final_score'
]
print('\nScore summary:')
print(df[summary_cols].describe().T)


Searching input files from roots:
 - /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/notebooks
 - /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding
 - /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook

Input matching file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step06_job_and_matching/07_candidate_job_matching.xlsx
Input semantic file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/12_candidate_job_semantic_similarity.parquet

Saved:
 - /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.parquet
 - /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/